In [3]:
import pygame
import random
import psycopg2
import sys

conn = psycopg2.connect(
    dbname="lab10",
    user="postgres",
    password="#Kazakhstan2050",
    host="localhost",
    port="5432"
)
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS users (
    id SERIAL PRIMARY KEY,
    username VARCHAR(50) UNIQUE NOT NULL
);
""")
cur.execute("""
CREATE TABLE IF NOT EXISTS user_score (
    id SERIAL PRIMARY KEY,
    user_id INTEGER REFERENCES users(id),
    level INTEGER DEFAULT 1,
    score INTEGER DEFAULT 0
);
""")
conn.commit()

username = input("Enter your username: ")
cur.execute("SELECT id FROM users WHERE username = %s", (username,))
user = cur.fetchone()

if user:
    user_id = user[0]
    cur.execute("SELECT level, score FROM user_score WHERE user_id = %s ORDER BY id DESC LIMIT 1", (user_id,))
    user_data = cur.fetchone()
    if user_data:
        level = user_data[0]
        score = user_data[1]
        speed = max(50, 200 - (level - 1) * 20)
    else:
        level = 1
        score = 0
        speed = 200
else:
    cur.execute("INSERT INTO users (username) VALUES (%s) RETURNING id", (username,))
    user_id = cur.fetchone()[0]
    conn.commit()
    level = 1
    score = 0
    speed = 200

pygame.init()
WIDTH, HEIGHT = 800, 600
WHITE, GREEN, BLUE, YELLOW, GRAY, BLACK, RED = (255, 255, 255), (0, 255, 0), (0, 0, 255), (255, 255, 0), (128, 128, 128), (0, 0, 0), (255, 0, 0)

screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Snake Game")

food_eaten, done, paused = False, False, False
head_square = [100, 100]
squares = [[30, 100], [40, 100], [50, 100], [60, 100], [70, 100], [80, 100], [90, 100], [100, 100]]
direction, next_dir = "right", "right"

FOOD_TIME_LIMIT = 30000
FOOD_TIMER_EVENT = pygame.USEREVENT + 1
food_spawn_time = pygame.time.get_ticks()
font = pygame.font.SysFont("times new roman", 20)

FOOD_TYPES = [
    {"color": GREEN, "points": 10, "probability": 0.6},
    {"color": YELLOW, "points": 20, "probability": 0.3},
    {"color": BLUE, "points": 50, "probability": 0.1}
]

def weighted_food_choice():
    return random.choices(FOOD_TYPES, weights=[f["probability"] for f in FOOD_TYPES])[0]

def spawn_food():
    global food_spawn_time, food_type
    while True:
        x, y = random.randrange(0, WIDTH // 10) * 10, random.randrange(0, HEIGHT // 10) * 10
        if [x, y] not in squares:
            pygame.time.set_timer(FOOD_TIMER_EVENT, FOOD_TIME_LIMIT)
            food_spawn_time = pygame.time.get_ticks()
            food_type = weighted_food_choice()
            return [x, y]

def save_score():
    cur.execute("INSERT INTO user_score (user_id, level, score) VALUES (%s, %s, %s)", (user_id, level, score))
    conn.commit()

def game_over():
    save_score()
    font_large = pygame.font.SysFont("times new roman", 45)
    text = font_large.render(f"Game Over! Score: {score}, Level: {level}", True, RED)
    screen.blit(text, (WIDTH // 4, HEIGHT // 2))
    pygame.display.update()
    pygame.time.delay(3000)
    cur.close()
    conn.close()
    pygame.quit()
    sys.exit()

food_pos = spawn_food()

while not done:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            save_score()
            cur.close()
            conn.close()
            done = True
        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_DOWN: next_dir = "down"
            if event.key == pygame.K_UP: next_dir = "up"
            if event.key == pygame.K_LEFT: next_dir = "left"
            if event.key == pygame.K_RIGHT: next_dir = "right"
            if event.key == pygame.K_p:
                paused = not paused
                if paused:
                    save_score()
        if event.type == FOOD_TIMER_EVENT:
            food_pos = spawn_food()

    if paused:
        continue

    if next_dir == "right" and direction != "left": direction = "right"
    if next_dir == "left" and direction != "right": direction = "left"
    if next_dir == "up" and direction != "down": direction = "up"
    if next_dir == "down" and direction != "up": direction = "down"

    if direction == "right": head_square[0] += 10
    if direction == "left": head_square[0] -= 10
    if direction == "up": head_square[1] -= 10
    if direction == "down": head_square[1] += 10

    if head_square[0] < 0 or head_square[0] >= WIDTH or head_square[1] < 0 or head_square[1] >= HEIGHT:
        game_over()
    if head_square in squares[:-1]:
        game_over()

    new_square = list(head_square)
    squares.append(new_square)

    if head_square == food_pos:
        food_eaten = True
        score += food_type["points"]
        if score % 50 == 0:
            level += 1
            speed = max(50, speed - 20)
    else:
        squares.pop(0)

    if food_eaten:
        food_pos = spawn_food()
        food_eaten = False

    elapsed_time = pygame.time.get_ticks() - food_spawn_time
    remaining_time = max(0, (FOOD_TIME_LIMIT - elapsed_time) // 1000)

    screen.fill(BLACK)
    screen.blit(font.render(f"Score: {score}", True, GRAY), (10, 10))
    screen.blit(font.render(f"Level: {level}", True, GRAY), (10, 30))
    screen.blit(font.render(f"Food Timer: {remaining_time}s", True, GRAY), (10, 50))

    pygame.draw.circle(screen, food_type["color"], (food_pos[0] + 5, food_pos[1] + 5), 5)
    for segment in squares:
        pygame.draw.rect(screen, WHITE, pygame.Rect(segment[0], segment[1], 10, 10))

    pygame.display.flip()
    pygame.time.delay(speed)

pygame.quit()
sys.exit()


Enter your username: nurdan


SystemExit: 